# Task 4: Commenter-Level Graph Analysis

**Project:** Coordinated Amplification and Misinformation Detection in Global YouTube Conflict Narratives  
**Course:** CS 418 (UIC)

This notebook builds a **commenter co-occurrence graph** from YouTube comments on Russia-Ukraine conflict videos:

1. **Bot-like accounts** via composite behavioral scoring
2. **Communities** of coordinated commenters via Louvain clustering
3. **Influential commenters** via PageRank

**Data engineering highlight:** All heavy aggregation (commenter stats, edge-list computation) is pushed server-side to BigQuery, avoiding the need to pull 1M+ raw comment rows into local memory.

## 1. Setup & Data Loading

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from utils.bq_helpers import query_to_df, table, COLORS

print('Libraries loaded successfully')

In [ ]:
# Server-side commenter statistics — aggregated in BigQuery, not locally
commenter_stats_sql = f"""
SELECT author_channel_id,
       COUNT(DISTINCT video_id) AS video_count,
       COUNT(*) AS comment_count,
       MIN(published_at) AS first_comment,
       MAX(published_at) AS last_comment,
       COUNT(DISTINCT DATE(published_at)) AS active_days
FROM (
  SELECT video_id, author_channel_id, published_at FROM {table('comment1')}
  UNION ALL SELECT video_id, author_channel_id, published_at FROM {table('comment2')}
  UNION ALL SELECT video_id, author_channel_id, published_at FROM {table('comment3')}
  UNION ALL SELECT video_id, author_channel_id, published_at FROM {table('comment4')}
  UNION ALL SELECT video_id, author_channel_id, published_at FROM {table('comment5')}
)
GROUP BY author_channel_id
HAVING author_channel_id IS NOT NULL
"""

print('Querying commenter stats from BigQuery (server-side aggregation)...')
commenter_stats = query_to_df(commenter_stats_sql)
print(f'Commenter stats: {commenter_stats.shape}')
print(f'\nDescriptive stats:')
commenter_stats[['video_count', 'comment_count', 'active_days']].describe()

## 2. Server-Side Co-Occurrence Edge List

### Why this matters for Data Engineering interviews

Computing commenter co-occurrence naively would require:
1. Pulling 1M+ comment rows into Python
2. Building an in-memory self-join (O(n^2) memory)

Instead, we **push the computation to BigQuery** — the warehouse handles the self-join on its distributed engine, and we only pull back the aggregated edge list. This is a fundamental data engineering pattern: **query pushdown**.

In [ ]:
# Server-side co-occurrence computation
edge_list_sql = f"""
WITH commenter_videos AS (
  SELECT DISTINCT author_channel_id, video_id
  FROM (
    SELECT author_channel_id, video_id FROM {table('comment1')}
    UNION ALL SELECT author_channel_id, video_id FROM {table('comment2')}
    UNION ALL SELECT author_channel_id, video_id FROM {table('comment3')}
    UNION ALL SELECT author_channel_id, video_id FROM {table('comment4')}
    UNION ALL SELECT author_channel_id, video_id FROM {table('comment5')}
  )
  WHERE author_channel_id IS NOT NULL
)
SELECT a.author_channel_id AS commenter_a,
       b.author_channel_id AS commenter_b,
       COUNT(DISTINCT a.video_id) AS shared_videos
FROM commenter_videos a
JOIN commenter_videos b
  ON a.video_id = b.video_id
  AND a.author_channel_id < b.author_channel_id
GROUP BY 1, 2
HAVING shared_videos >= 3
"""

print('Computing co-occurrence edge list in BigQuery (server-side self-join)...')
print('This may take a few minutes for 1M+ comments...')
edge_list = query_to_df(edge_list_sql)
print(f'\nEdge list: {edge_list.shape}')
print(f'Shared videos stats:')
print(edge_list['shared_videos'].describe())

## 3. Network Construction

In [ ]:
# Build NetworkX graph from edge list
G = nx.Graph()

for _, row in tqdm(edge_list.iterrows(), total=len(edge_list), desc='Building graph'):
    G.add_edge(row['commenter_a'], row['commenter_b'], weight=row['shared_videos'])

# Add node attributes from commenter stats
stats_dict = commenter_stats.set_index('author_channel_id').to_dict('index')
for node in G.nodes():
    if node in stats_dict:
        for key, val in stats_dict[node].items():
            G.nodes[node][key] = val

print(f'Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'Density: {nx.density(G):.6f}')
print(f'Connected components: {nx.number_connected_components(G)}')

# Largest connected component
largest_cc = max(nx.connected_components(G), key=len)
print(f'Largest connected component: {len(largest_cc):,} nodes')

## 4. Bot Detection

Composite bot score based on behavioral heuristics:
- **High volume score** (0.3): Percentile rank of video_count
- **Burst score** (0.3): comment_count / active_days > 20
- **Cross-channel score** (0.2): video_count / active_days > 5
- **Network score** (0.2): Degree centrality (highly connected commenters)

In [ ]:
# Get degree centrality for nodes in graph
degree_cent = nx.degree_centrality(G)

# Build bot scoring DataFrame
bot_df = commenter_stats.copy()
bot_df['active_days'] = bot_df['active_days'].clip(lower=1)  # avoid division by zero

# High volume score (normalized percentile rank of video_count)
bot_df['high_volume_score'] = bot_df['video_count'].rank(pct=True) * 0.3

# Burst score (comments per active day)
bot_df['comments_per_day'] = bot_df['comment_count'] / bot_df['active_days']
bot_df['burst_score'] = np.clip(bot_df['comments_per_day'] / 20, 0, 1) * 0.3

# Cross-channel score (videos per active day)
bot_df['videos_per_day'] = bot_df['video_count'] / bot_df['active_days']
bot_df['cross_channel_score'] = np.clip(bot_df['videos_per_day'] / 5, 0, 1) * 0.2

# Network score (degree centrality from commenter graph)
bot_df['degree_centrality'] = bot_df['author_channel_id'].map(degree_cent).fillna(0)
max_dc = bot_df['degree_centrality'].max()
bot_df['network_score'] = (bot_df['degree_centrality'] / max_dc * 0.2) if max_dc > 0 else 0

# Composite bot score
bot_df['bot_score'] = (
    bot_df['high_volume_score'] +
    bot_df['burst_score'] +
    bot_df['cross_channel_score'] +
    bot_df['network_score']
)

bot_df['is_likely_bot'] = (bot_df['bot_score'] >= 0.5).astype(int)

n_bots = bot_df['is_likely_bot'].sum()
print(f'Total commenters: {len(bot_df):,}')
print(f'Likely bots (score >= 0.5): {n_bots:,} ({n_bots/len(bot_df)*100:.1f}%)')
print(f'\nBot score distribution:')
print(bot_df['bot_score'].describe())

In [ ]:
# Bot score distribution
fig = px.histogram(bot_df, x='bot_score', nbins=50,
    title='Distribution of Composite Bot Scores',
    color_discrete_sequence=[COLORS['primary']])
fig.add_vline(x=0.5, line_dash='dash', line_color='red',
    annotation_text='Bot threshold (0.5)')
fig.update_layout(height=400, width=900, xaxis_title='Bot Score', yaxis_title='Count')
fig.show()

In [ ]:
# Top 20 suspected bots
top_bots = bot_df.nlargest(20, 'bot_score')
print('Top 20 Suspected Bots:')
top_bots[['author_channel_id', 'comment_count', 'video_count', 'active_days',
          'comments_per_day', 'bot_score']].reset_index(drop=True)

In [ ]:
# Activity scatter: comment count vs video count
sample = bot_df.sample(min(10000, len(bot_df)), random_state=42)
fig = px.scatter(sample, x='video_count', y='comment_count',
    color='bot_score', color_continuous_scale='RdYlBu_r',
    title='Commenter Activity (colored by bot score)',
    log_x=True, log_y=True, opacity=0.4,
    labels={'video_count': 'Videos Commented On', 'comment_count': 'Total Comments'})
fig.update_layout(height=550, width=900)
fig.show()

## 5. Community Detection

Louvain clustering on top-5000 commenters by degree for tractable community detection.

In [ ]:
# Take subgraph of top commenters by degree
top_n = min(5000, G.number_of_nodes())
top_nodes = sorted(G.nodes(), key=lambda n: G.degree(n), reverse=True)[:top_n]
G_sub = G.subgraph(top_nodes).copy()

print(f'Subgraph for clustering: {G_sub.number_of_nodes():,} nodes, {G_sub.number_of_edges():,} edges')

# Louvain community detection
partition = community_louvain.best_partition(G_sub, weight='weight', random_state=42)
modularity = community_louvain.modularity(partition, G_sub, weight='weight')

n_communities = len(set(partition.values()))
print(f'Communities found: {n_communities}')
print(f'Modularity: {modularity:.4f}')

# Community sizes
comm_sizes = pd.Series(partition).value_counts().reset_index()
comm_sizes.columns = ['cluster_id', 'size']
print(f'\nTop 10 community sizes:')
print(comm_sizes.head(10).to_string())

In [ ]:
# Community size bar chart
fig = px.bar(comm_sizes.head(20), x='cluster_id', y='size',
    title='Top 20 Commenter Communities by Size',
    labels={'cluster_id': 'Community', 'size': 'Members'})
fig.update_layout(height=400, width=900)
fig.show()

## 6. Cross-Channel Analysis

Check if commenter clusters map to channel clusters from Task 2.

In [ ]:
# Try to load channel clusters from Task 2
try:
    channel_clusters = pd.read_csv('../task2_network_analysis/outputs/channel_clusters.csv')
    print(f'Loaded channel clusters: {channel_clusters.shape}')
    
    # Load commenter-to-video mapping for cross-reference
    cross_sql = f"""
    SELECT DISTINCT c.author_channel_id, v.channel_id
    FROM (
      SELECT author_channel_id, video_id FROM {table('comment1')}
      UNION ALL SELECT author_channel_id, video_id FROM {table('comment2')}
      UNION ALL SELECT author_channel_id, video_id FROM {table('comment3')}
      UNION ALL SELECT author_channel_id, video_id FROM {table('comment4')}
      UNION ALL SELECT author_channel_id, video_id FROM {table('comment5')}
    ) c
    JOIN {table('video_data')} v ON c.video_id = v.video_id
    WHERE c.author_channel_id IS NOT NULL
    """
    commenter_channels = query_to_df(cross_sql)
    print(f'Commenter-channel links: {len(commenter_channels):,}')
    
    # Merge with channel clusters
    commenter_channels = commenter_channels.merge(
        channel_clusters[['channel_id', 'cluster_id']],
        on='channel_id', how='left'
    )
    
    # For each commenter community, what channel clusters do they comment on?
    commenter_channels['commenter_cluster'] = commenter_channels['author_channel_id'].map(partition)
    cross = commenter_channels.dropna(subset=['commenter_cluster', 'cluster_id'])
    if len(cross) > 0:
        cross_tab = pd.crosstab(cross['commenter_cluster'].astype(int),
                                cross['cluster_id'].astype(int), normalize='index')
        print(f'\nCross-tabulation (commenter community vs channel cluster):')
        print(cross_tab.head(10))

except FileNotFoundError:
    print('Channel clusters not found. Run Task 2 notebook first.')
    print('Skipping cross-channel analysis.')

## 7. Top Influencers — PageRank

In [ ]:
# PageRank on commenter subgraph
print('Computing PageRank...')
pr = nx.pagerank(G_sub, weight='weight')

# Build leaderboard
leaderboard = pd.DataFrame({
    'author_channel_id': list(pr.keys()),
    'pagerank': list(pr.values()),
}).sort_values('pagerank', ascending=False)

# Merge with stats and bot scores
leaderboard = leaderboard.merge(
    bot_df[['author_channel_id', 'comment_count', 'video_count', 'active_days', 'bot_score']],
    on='author_channel_id', how='left'
)
leaderboard['cluster_id'] = leaderboard['author_channel_id'].map(partition)
leaderboard['degree'] = leaderboard['author_channel_id'].map(lambda n: G_sub.degree(n) if n in G_sub else 0)

print('\nTop 20 Influential Commenters:')
top20 = leaderboard.head(20).reset_index(drop=True)
top20.index += 1
top20[['author_channel_id', 'comment_count', 'video_count', 'pagerank', 'bot_score', 'cluster_id']]

In [ ]:
# PageRank bar chart
fig = px.bar(top20, x='pagerank', y='author_channel_id', orientation='h',
    color='bot_score', color_continuous_scale='RdYlBu_r',
    title='Top 20 Commenters by PageRank (colored by bot score)')
fig.update_layout(height=600, width=1000, yaxis=dict(autorange='reversed'))
fig.show()

## 8. Save Outputs

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)

# Save bot scores
bot_output = bot_df[['author_channel_id', 'comment_count', 'video_count',
                      'active_days', 'bot_score', 'is_likely_bot']]
bot_output.to_csv('outputs/commenter_bot_scores.csv', index=False)
print(f'Saved outputs/commenter_bot_scores.csv ({len(bot_output):,} rows)')

# Save commenter clusters
cluster_output = leaderboard[['author_channel_id', 'cluster_id', 'pagerank', 'degree']]
cluster_output.to_csv('outputs/commenter_clusters.csv', index=False)
print(f'Saved outputs/commenter_clusters.csv ({len(cluster_output):,} rows)')

print('\nDone!')